# Full training and analysis

The cluster-scale counterpart of [01_downscale_training.ipynb](01_downscale_training.ipynb). This notebook reviews settings, prepares data, launches the Python trainer, and analyzes saved checkpoints. Training implementation stays in `src/fox_experiments/full_training/`.

**Target:** the default config is an approximately 124M-parameter model, 2,048-token training windows, BF16, and four H200-class GPUs on one node. It uses the supplied paper's fused forgetting-attention kernel with pruning disabled. This is a scaled optimizer experiment, not the paper's exact 760M / 48B-token recipe.

You can inspect this notebook on Colab. Run the full job from Jupyter on your allocated cluster node, or use the Slurm script in `scripts/`. A Colab T4 and the 18 MiB pilot dataset are insufficient for this configuration.

No full download or GPU training starts until its switch is enabled. Read [full-training instructions](../docs/full_training.md) and [scientific design](../docs/experiment_design.md).

## 1. Open the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # After publishing: https://github.com/YOUR_NAME/fox_experiments.git
REPO_REF = "main"  # Branch, tag, or commit to use in a new Colab clone.
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/fox_experiments")]
REPO = next((p for p in candidates if (p / "src/fox_experiments").is_dir()), None)
if REPO is None:
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or open this notebook inside a local clone.")
    REPO = Path("/content/fox_experiments")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REPO_REF])
REPO = REPO.resolve()
os.chdir(REPO)
print("Repository:", REPO)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
      if (REPO / ".git/HEAD").exists() and subprocess.run(
          ["git", "rev-parse", "--verify", "HEAD"], capture_output=True).returncode == 0
      else "No commit yet: commit the source before a scientific run.")

## 2. Install the full-training dependencies

In [ ]:
if os.environ.get("FOX_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[full]"])
sys.path.insert(0, str(REPO / "src"))

import json
import torch
import pandas as pd
from IPython.display import display, Image
print("PyTorch:", torch.__version__)
print("Visible GPUs:", torch.cuda.device_count())

## 3. Choose the experiment and storage

Edit `configs/full/h200_124m.json` or copy it to a named config. Keep rates, epsilon schedule, training lags and evaluation grids fixed after short-only validation. `SEEDS` controls independent training runs; gate/optimizer branches within a seed share an acquired function. The complete matrix has two acquisition sources and nine continuation branches per seed.

Use cluster scratch/storage paths for data and outputs. This notebook's launch command expects all GPUs on one node. Use the Slurm entry point for an allocation that survives a notebook disconnect.

In [ ]:
from fox_experiments.full_training.config import load_config, budget

CONFIG_PATH = REPO / "configs/full/h200_124m.json"
DATA_ROOT = Path(os.environ.get("FOX_FULL_DATA_DIR", "/data/longcrawl64"))
RUN_ROOT = Path(os.environ.get("FOX_FULL_RUN_DIR", str(REPO / "outputs/full/h200_124m_v1")))
SEEDS = [0, 1, 2]
CONFIG = load_config(CONFIG_PATH)
CONFIG.data_root = str(DATA_ROOT)
GPUS = CONFIG.expected_world_size

display(pd.Series(budget(CONFIG), name="Per-seed budget").to_frame())
print("All-seed input tokens:", len(SEEDS) * budget(CONFIG)["two_sources_nine_arms_tokens"])
print("Outputs:", RUN_ROOT)

In [ ]:
from fox_experiments.full_training.__main__ import preflight

preview = preflight(CONFIG)
print("Parameters:", preview["factorized_parameter_count"])
print("Adam parameter/gradient/moment GiB per rank:",
      round(preview["adam_parameter_gradient_moment_GiB_per_rank"], 2))
print(preview["memory_note"])
display(pd.Series(CONFIG.task, name="Task / optimizer setting").to_frame())

## 4. Prepare the native full corpus

Training uses `train.zarr`; development and test use disjoint regions of `heldout.zarr`. The pilot's source-heldout training subset is not used here. Provision roughly **1 TB** of data storage, plus checkpoints. The download is version-pinned and resumes existing files. If your cluster already has these two native stores, point `DATA_ROOT` there.

The loader rejects missing chunks rather than treating them as zero tokens. Source rows are 65,536-token storage blocks, not guaranteed single web documents.

In [ ]:
from fox_experiments.full_training.data import download_plan, download

DOWNLOAD_FULL_DATA = False
print(json.dumps(download_plan(DATA_ROOT), indent=2))
if DOWNLOAD_FULL_DATA:
    download(DATA_ROOT)

## 5. Check the GPU kernel and complete data

Enable this on the allocated GPU node before launching. The check compares fused attention outputs and Q/K/V/gate gradients against the dense reference, then checks native corpus completeness. These GPU checks have not been run on the development machine.

Keep the same checkpoint and data when comparing implementations. BF16 and distributed execution are practical changes from the FP32 Colab pilot.

In [ ]:
RUN_HARDWARE_CHECKS = False
if RUN_HARDWARE_CHECKS:
    checks = preflight(CONFIG, check_data=True, check_kernel=True)
    print(json.dumps(checks["kernel"], indent=2))
    print("Native data stores validated.")

## 6. Review the paired training plan

Within each seed, factorized/direct first gates branch from the same direct-gate source; original data-dependent FoX has its own source. Each continuation starts with fresh optimizer moments. It then retains those moments through checkpoint resumes. Source acquisition mixes natural LM training and short latest-write training.

Full-scale rates are explicit starting candidates. Calibrate candidates with the short-only `evaluate --split tune` workflow in the full-training guide, then freeze the chosen rates in a new config before launching this matrix. Long-test accuracy must not select a rate.

The commands train each branch and evaluate its middle and final checkpoints. Source baselines are saved separately. They run sequentially using the requested GPUs.

In [ ]:
import shlex
from fox_experiments.full_training.__main__ import matrix_commands

commands = matrix_commands(CONFIG, CONFIG_PATH, SEEDS, GPUS, RUN_ROOT)
print("Training / evaluation commands:", len(commands))
print("First training command:", shlex.join(commands[0]))
launch = [sys.executable, "-m", "fox_experiments.full_training", "matrix",
          "--config", str(CONFIG_PATH), "--data-root", str(DATA_ROOT),
          "--run-root", str(RUN_ROOT), "--gpus", str(GPUS),
          "--seeds", *map(str, SEEDS)]
print("CLI preview:", shlex.join(launch))

## 7. Launch or resume

Set `RUN_TRAINING=True` on the allocated node after reviewing the budget and freezing short-validated settings. The launch rechecks data and the CUDA kernel. Repeating the same command resumes compatible checkpoints, including optimizer, schedule and random-number states. Changing scientific settings or source weights requires a new output directory.

For unattended jobs, copy the printed CLI command into the supplied Slurm workflow. Notebook subprocesses need the notebook runtime to stay alive.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    preflight(CONFIG, check_data=True, check_kernel=True)
    subprocess.run(launch + ["--execute"], check=True)
else:
    print("Plan only. Training has not been launched.")

## 8. Analyze completed test evaluations

Set `RUN_ROOT` to an existing experiment to run this section without training. Failed or incomplete runs should be inspected alongside successful ones. The combined report uses continuation branches for optimizer comparisons and keeps source acquisition baselines separately.

Look for short-rule retention first, then target-lag curves across fixed old-prefix counts and progression across checkpoints. Both optimizers passing at the largest tested lag means the boundary remains unobserved.

In [ ]:
from fox_experiments.full_training.evaluation import collect_reports

report = collect_reports(RUN_ROOT)
print(report)
ANALYSIS = RUN_ROOT / "analysis"
if (ANALYSIS / "summary.csv").exists():
    display(pd.read_csv(ANALYSIS / "summary.csv"))

In [ ]:
from fox_experiments.evaluation import summarize_results

if (ANALYSIS / "retrieval_raw.csv").exists():
    panels, ranges, contrasts, figures = summarize_results(ANALYSIS)
    display(ranges)
    display(contrasts.head(20))
    for figure in figures:
        display(Image(filename=str(figure)))
else:
    print("No saved test predictions yet. Run the matrix or point RUN_ROOT to completed results.")

## 9. Export the report

Exports reports and figures only; full model/optimizer checkpoints stay in their run directories. Keep the resolved config, raw predictions, source identity and failed-run logs with the final study.

Theoretical arbitrary-prefix diagnostics belong to the controlled model in the downscale notebook. Finite full-model evaluations do not certify infinite-distance retrieval.

In [ ]:
from fox_experiments.notebook_utils import archive_results

if (ANALYSIS / "summary.csv").exists():
    archive = archive_results(RUN_ROOT / "analysis.zip", {"analysis": ANALYSIS})
    print("Saved:", archive)